# XGBoost Classification: Ataxia (1) vs PD (2)
Using Xsens gait features from `All_Results_T0T1_Gait_VV_CR_20260303.csv`

In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("icc_py/All_Results_T0T1_Gait_VV_CR_20260303.csv")
print(f"Shape: {df.shape}")
print(f"\n1ATX2PD value counts:\n{df['1ATX2PD'].value_counts()}")
df.head()

Shape: (71, 209)

1ATX2PD value counts:
1ATX2PD
1    44
2    27
Name: count, dtype: int64


,ID,1ATX2PD,Group,Control,Group_smaller,JT_T1_NG.cadence,JT_T1_NG.stridetime,JT_T1_NG.stridetimeSD,JT_T1_NG.v,JT_T1_RG.cadence,...,Myo_T2_RG.Kadenz,Myo_T2_RG.KadenzSD,Myo_T2_RG.Geschwindigkeit,Myo_T2_RG.GeschwindigkeitSD,Myo_T2_RG.FußrotationR,Myo_T2_RG.FußrotationR_SD,Myo_T2_RG.FußrotationL,Myo_T2_RG.FußrotationL_SD,Myo_T2_RG.Schrittbreite,Myo_T2_RG.SchrittbreiteSD
0,101,1,2.0,0,2,1.139055,1.71596,0.605090,0.33301,1.159913,...,1.266692,0.116669,0.277780,0.055556,0.8,2.8,2.3,2.5,24.0,2.0
1,102,2,2.0,1,1,NaN,NaN,NaN,NaN,NaN,...,1.816703,0.166670,0.611116,0.166668,-2.6,6.0,-5.8,4.1,19.0,6.0
2,103,2,3.0,1,1,NaN,NaN,NaN,NaN,1.114887,...,1.800036,0.166670,0.611116,0.083334,12.6,4.8,3.5,6.1,13.0,3.0
3,104,2,2.0,0,2,1.706671,1.17094,0.074975,0.82105,1.451326,...,1.550031,0.100002,0.611116,0.083334,-10.9,1.5,-10.8,4.2,18.0,3.0
4,105,2,3.0,0,3,1.299494,1.51670,0.086989,0.49527,1.180258,...,1.500030,0.116669,0.361114,0.083334,-9.0,3.6,-2.5,4.3,22.0,3.0


In [3]:
# Select only Xsens features
xsens_cols = [c for c in df.columns if c.startswith("Xsens_")]
print(f"Number of Xsens features: {len(xsens_cols)}")

# Target: 1ATX2PD (1=Ataxia, 2=PD) — encode to 0/1 for XGBoost
y = df["1ATX2PD"].values  # 1=Ataxia, 2=PD
X = df[xsens_cols].copy()

# Check missingness per row
missing_pct = X.isnull().mean(axis=1) * 100
print(f"\nMissing % per subject: min={missing_pct.min():.1f}%, max={missing_pct.max():.1f}%, median={missing_pct.median():.1f}%")

# Drop subjects with >80% missing Xsens data
keep_mask = missing_pct < 80
print(f"\nKeeping {keep_mask.sum()}/{len(keep_mask)} subjects (dropping {(~keep_mask).sum()} with >80% missing)")
X = X[keep_mask].reset_index(drop=True)
y = y[keep_mask]

# Encode target: Ataxia=0, PD=1
y_encoded = (y - 1).astype(int)  # 1->0 (Ataxia), 2->1 (PD)
print(f"\nFinal class distribution: Ataxia(0)={sum(y_encoded==0)}, PD(1)={sum(y_encoded==1)}")

Number of Xsens features: 132

Missing % per subject: min=0.0%, max=100.0%, median=0.8%

Keeping 70/71 subjects (dropping 1 with >80% missing)

Final class distribution: Ataxia(0)=43, PD(1)=27


## Leave-One-Out Cross-Validation (LOOCV)
Small sample size → LOOCV gives least-biased estimate of generalization performance.

In [4]:
# XGBoost with LOOCV
# Conservative hyperparameters for small sample size
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.5,
    reg_alpha=1.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

loo = LeaveOneOut()

# LOOCV predictions
y_pred = cross_val_predict(model, X, y_encoded, cv=loo, method="predict")
y_prob = cross_val_predict(model, X, y_encoded, cv=loo, method="predict_proba")[:, 1]

print("=== LOOCV Results ===")
print(f"\nClassification Report (0=Ataxia, 1=PD):")
print(classification_report(y_encoded, y_pred, target_names=["Ataxia", "PD"]))

auc = roc_auc_score(y_encoded, y_prob)
print(f"ROC AUC: {auc:.3f}")

accuracy = (y_pred == y_encoded).mean()
print(f"Overall Accuracy: {accuracy:.3f}")

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Xsens_T2_NG.AvgLFstancetime: object, Xsens_T1_RG.AvgLFstancetime: object

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_encoded, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Ataxia", "PD"])
disp.plot(ax=ax, cmap="Blues")
ax.set_title("LOOCV Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix:")
print(f"  Ataxia correctly classified: {cm[0,0]}/{cm[0].sum()}")
print(f"  PD correctly classified: {cm[1,1]}/{cm[1].sum()}")

## Feature Importance
Fit on full dataset to see which Xsens features are most discriminative.

In [ ]:
# Fit on full data for feature importance
model.fit(X, y_encoded)

# Top 20 features by gain
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=xsens_cols).sort_values(ascending=False)
top20 = feat_imp.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20.sort_values().plot.barh(ax=ax)
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("Top 20 Xsens Features for Ataxia vs PD Classification")
plt.tight_layout()
plt.show()

print("\nTop 20 features:")
for i, (feat, imp) in enumerate(top20.items(), 1):
    print(f"  {i:2d}. {feat}: {imp:.4f}")

## ROC Curve

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_encoded, y_prob, name="XGBoost LOOCV", ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_title(f"ROC Curve (AUC = {auc:.3f})")
plt.tight_layout()
plt.show()

## Stratified 5-Fold CV (for comparison)

In [ ]:
# Stratified 5-fold CV as a secondary check
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_5f = cross_val_predict(model, X, y_encoded, cv=skf, method="predict")
y_prob_5f = cross_val_predict(model, X, y_encoded, cv=skf, method="predict_proba")[:, 1]

print("=== Stratified 5-Fold CV Results ===")
print(classification_report(y_encoded, y_pred_5f, target_names=["Ataxia", "PD"]))

auc_5f = roc_auc_score(y_encoded, y_prob_5f)
acc_5f = (y_pred_5f == y_encoded).mean()
print(f"ROC AUC: {auc_5f:.3f}")
print(f"Accuracy: {acc_5f:.3f}")